In [1]:
!pip install anthropic pypdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.6 MB/s eta 0:00:00


In [2]:
import anthropic
import PyPDF2
import io
from google.colab import files

# Upload JD and CV
print("Upload your Job Description PDF first, then your CV PDF")
uploaded = files.upload()
print(f"Uploaded: {list(uploaded.keys())}")

Upload your Job Description PDF first, then your CV PDF


Saving jd.pdf to jd.pdf
Uploaded: ['jd.pdf']


In [6]:
!pip install pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 55.4 MB/s eta 0:00:00


In [7]:
import fitz

# Re-extract JD with pymupdf
for filename, content in uploaded.items():
    pdf_document = fitz.open(stream=content, filetype="pdf")
    text = ""
    for page in pdf_document:
        text += page.get_text()
    docs[filename] = text
    print(f"✓ {filename}: {len(text)} characters")

✓ jd.pdf: 0 characters


In [11]:
claude = anthropic.Anthropic(api_key="YOUR_API_KEY_HERE")

def analyze_candidate(jd, cv):
    response = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2000,
        messages=[{
            "role": "user",
            "content": f"""You are an expert HR recruiter in Singapore. Analyze this candidate against the job description.

JOB DESCRIPTION:
{jd}

CANDIDATE CV:
{cv}

Provide your analysis in this exact format:

## OVERALL FIT SCORE: X/10

## COMPETENCY SCORES
- Technical skills: X/10
- Experience match: X/10
- Education fit: X/10
- Culture/soft skills: X/10

## STRENGTHS (top 3)
-
-
-

## GAPS (top 3)
-
-
-

## INTERVIEW QUESTIONS (5 questions targeting the gaps)
1.
2.
3.
4.
5.

## HIRING RECOMMENDATION
One sentence recommendation."""
        }]
    )
    print(response.content[0].text)

analyze_candidate(jd_text, cv_text)

## OVERALL FIT SCORE: 6.5/10

## COMPETENCY SCORES
- Technical skills: 7/10
- Experience match: 6/10
- Education fit: 5/10
- Culture/soft skills: 7/10

## STRENGTHS (top 3)
- **Strong multi-jurisdiction APAC operational expertise**: 11+ years across Singapore, Vietnam, India, Indonesia, and Europe with proven ability to build HR foundations, manage compliance, and navigate employment law across diverse markets—directly aligned with the regional scope required.
- **Demonstrated operational excellence and process design**: Built scalable HR systems from scratch, automated workflows (BoldSign, Info-Tech), and established regional playbooks that reduced turnaround times by 60%—shows the ability to embed efficiency and structure in growing organisations.
- **People Operations generalist capability**: Proven experience spanning employee relations, performance management, compliance, payroll, and operational HR partnering with senior leaders across multiple functions and markets—meets the gen

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
!pip install python-docx

import docx

def extract_text_from_docx(filepath):
    doc = docx.Document(filepath)
    return "\n".join([para.text for para in doc.paragraphs])

# Test it
cv_folder = '/content/drive/MyDrive/HR-AI-Colabs-github-portfolio/recruiter-assistant/cvs'
cv_files = [f for f in os.listdir(cv_folder) if f.endswith('.docx')]
print(f"Found {len(cv_files)} CVs: {cv_files}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 1.7 MB/s eta 0:00:00
Found 3 CVs: ['Resume - Ng Thian Kiat.docx', 'do not use - Aprille Resume.docx', 'OTR Resume - Cedric Bonnard.docx']


In [30]:
import re

def clean_and_parse_json(text):
    text = re.sub(r'```json|```', '', text).strip()
    return json.loads(text)

def get_cv_text(filename):
    filepath = os.path.join(cv_folder, filename)
    try:
        return extract_text_from_docx(filepath)
    except:
        return ""

print("Functions defined")

Functions defined


In [37]:
def analyze_multiple_candidates(jd):
    results = []
    all_files = [f for f in os.listdir(cv_folder)
                 if (f.endswith('.docx') or f.endswith('.pdf'))
                 and 'do not use' not in f.lower()]
    print(f"Found {len(all_files)} CVs")

    for filename in all_files:
        cv_text = get_cv_text(filename)
        if len(cv_text) < 100:
            print(f"Skipping {filename}")
            continue
        print(f"Analyzing {filename}...")
        prompt = "Return ONLY raw JSON no markdown: {\"name\": \"x\", \"overall_score\": 7, \"technical\": 6, \"experience\": 7, \"education\": 5, \"soft_skills\": 8, \"top_strength\": \"x\", \"top_gap\": \"x\", \"recommendation\": \"hire/maybe/pass\"} JD: " + jd[:1500] + " CV: " + cv_text[:1500]
        response = claude.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}]
        )
        try:
            result = clean_and_parse_json(response.content[0].text)
            result['filename'] = filename
            results.append(result)
            print(f"✓ {result['name']}: {result['overall_score']}/10 - {result['recommendation']}")
        except:
            print(f"✗ {filename}: {response.content[0].text[:100]}")

    results.sort(key=lambda x: x['overall_score'], reverse=True)
    print(f"\n{'='*65}")
    print(f"{'RANK':<5} {'NAME':<25} {'SCORE':<7} {'REC':<8} {'TOP GAP'}")
    print(f"{'-'*65}")
    for i, r in enumerate(results, 1):
        print(f"{i:<5} {r['name']:<25} {r['overall_score']}/10  {r['recommendation']:<8} {r['top_gap'][:25]}")
    return results

results = analyze_multiple_candidates(jd_text)

Found 6 CVs
Analyzing Resume - Ng Thian Kiat.docx...
✓ Hospitality Professional: 1/10 - pass
Skipping OTR Resume - Cedric Bonnard.docx
Analyzing Celine Ong - Resume.pdf...
✓ Celine Ong: 3/10 - pass
Analyzing Resume - Barnabus Ng.docx...
✓ Recent Graduate Candidate: 2/10 - pass
Analyzing Copy of Elwin Tan - Resume.docx...
✓ Candidate: 2/10 - pass
Analyzing Resume - Daryl Neo.pdf...
✓ Daryl Neo: 2/10 - pass

RANK  NAME                      SCORE   REC      TOP GAP
-----------------------------------------------------------------
1     Celine Ong                3/10  pass     No People/HR experience; 
2     Recent Graduate Candidate 2/10  pass     No relevant HR/People Ope
3     Candidate                 2/10  pass     No HR/People Operations b
4     Daryl Neo                 2/10  pass     No HR, People Operations,
5     Hospitality Professional  1/10  pass     No HR, People Operations,


In [38]:
results = analyze_multiple_candidates(jd_text)


Found 6 CVs
Analyzing Resume - Ng Thian Kiat.docx...
✓ Hospitality Professional: 1/10 - pass
Skipping OTR Resume - Cedric Bonnard.docx
Analyzing Celine Ong - Resume.pdf...
✓ Celine Ong: 2/10 - pass
Analyzing Resume - Barnabus Ng.docx...
✓ Recent Graduate Candidate: 2/10 - pass
Analyzing Copy of Elwin Tan - Resume.docx...
✓ Candidate: 2/10 - pass
Analyzing Resume - Daryl Neo.pdf...
✓ Daryl Neo: 2/10 - pass

RANK  NAME                      SCORE   REC      TOP GAP
-----------------------------------------------------------------
1     Celine Ong                2/10  pass     No People/HR experience, 
2     Recent Graduate Candidate 2/10  pass     Lacks HR/People operation
3     Candidate                 2/10  pass     No People/HR strategy exp
4     Daryl Neo                 2/10  pass     Completely misaligned for
5     Hospitality Professional  1/10  pass     No HR, People Operations,
